## Data Preprocessing for ML(Supervised Model - Classification)

In [ ]:
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option('display.max_columns', None)

In [ ]:
# Reading Data
df = pd.read_parquet('Clients_LoanLens_clean.parquet')
print(df.shape)
df.head(5)

In [ ]:
df['NAME_CONTRACT_STATUS'].isna().sum()

In [ ]:
# Clients with no previous application have no target → not usable for the approval model
df = df.dropna(subset=['NAME_CONTRACT_STATUS'])

print(df.shape)
print(df['NAME_CONTRACT_STATUS'].isna().sum())

- **No target:** dropped 17,906 clients with no previous application (no Approved / Refused decision). They stay in the clean file for the segmentation notebook.

- Check Target Column (Imbalanced)

In [ ]:
print(df['NAME_CONTRACT_STATUS'].value_counts())
print(df['NAME_CONTRACT_STATUS'].value_counts(normalize=True).round(4) * 100)

- The target is **imbalanced**: 84.6% Approved vs 15.4% Refused. Accuracy alone would be misleading (predicting "Approved" for everyone gives 85%), so we will focus on **Precision, Recall and F1 for the Refused class**.

In [ ]:
ratio = df['AMT_CREDIT'] / df['AMT_APPLICATION']
ratio.groupby(df['NAME_CONTRACT_STATUS']).describe().round(2)

In [ ]:
# Data leakage: AMT_CREDIT is set by the bank with the decision (credit/application ratio differs
# between Approved and Refused), and AMT_ANNUITY is calculated from it and the bank's interest rate
df.drop(columns=['AMT_CREDIT', 'AMT_ANNUITY'], inplace=True)

df.shape

- **Data leakage:** `AMT_CREDIT` (approved amount) is set by the bank with the decision: its ratio to the requested amount differs between Approved (median 1.00) and Refused (median 1.07). `AMT_ANNUITY` is calculated from it and the bank's interest rate. Both dropped.

In [ ]:
df['NAME_CONTRACT_STATUS'] = df['NAME_CONTRACT_STATUS'].map({'Approved': 0, 'Refused': 1})
df['NAME_CONTRACT_STATUS'].value_counts()

In [ ]:
print(df[['AMT_APPLICATION', 'AMT_GOODS_PRICE']].corr().round(4))
print(((df['AMT_APPLICATION'] == df['AMT_GOODS_PRICE']).mean() * 100).round(2))

In [ ]:
# AMT_GOODS_PRICE = AMT_APPLICATION in 99.99% of applications (correlation = 1.0) → redundant
df.drop(columns=['AMT_GOODS_PRICE'], inplace=True)

df.shape

- **AMT_GOODS_PRICE:** equal to `AMT_APPLICATION` in 99.99% of applications (correlation = 1.0) → dropped as redundant.

## Feature Engineering

In [ ]:
# How many times the requested amount is the client's income
df['LOAN_TO_INCOME'] = df['AMT_APPLICATION'] / df['AMT_INCOME_TOTAL']
# Share of the amount paid upfront
df['DOWN_PAYMENT_RATIO'] = df['AMT_DOWN_PAYMENT'] / df['AMT_APPLICATION']
# Monthly installment / income (revolving loans have no fixed term → 0)
df['PAYMENT_TO_INCOME'] = (df['AMT_APPLICATION'] / df['CNT_PAYMENT'].replace(0, np.nan)) / df['AMT_INCOME_TOTAL']
df['PAYMENT_TO_INCOME'] = df['PAYMENT_TO_INCOME'].fillna(0)

- **New features:** `LOAN_TO_INCOME`, `DOWN_PAYMENT_RATIO`, `PAYMENT_TO_INCOME` (CNT_PAYMENT = 0 for 33,779 revolving loans → 0). They did not change F1 (tree models already learn these relations from the original columns) but are kept for business interpretation.

### Data Splitting & Preprocessing for ML

In [ ]:
x = df.drop(columns=['NAME_CONTRACT_STATUS'])
y = df['NAME_CONTRACT_STATUS']

print(x.shape)
print(y.shape)

- Numerical Columns

In [ ]:
num_cols = x.select_dtypes(include= 'number').columns
num_cols

In [ ]:
log_cols = ['AMT_INCOME_TOTAL', 'AMT_APPLICATION', 'AMT_DOWN_PAYMENT']

no_scaling_cols = ['FLAG_DOCUMENT_3', 'FLAG_DOCUMENT_5', 'FLAG_DOCUMENT_6', 'FLAG_DOCUMENT_8',
                   'FLAG_EMP_PHONE', 'LIVE_REGION_NOT_WORK_REGION', 'HAS_BUREAU_RECORD']

scal_cols = ['CNT_CHILDREN', 'CNT_FAM_MEMBERS', 'OWN_CAR_AGE', 'REGION_POPULATION_RELATIVE', 'REGION_RATING_CLIENT_W_CITY',
             'EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3', 'CNT_PAYMENT', 'OTHER_DOCUMENTS', 'TOTAL_BUREAU_REQ',
             'MONTH_EMPLOYED', 'AGE_YEARS', 'LOAN_TO_INCOME', 'DOWN_PAYMENT_RATIO', 'PAYMENT_TO_INCOME']

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import RobustScaler

num_pipeline = Pipeline(steps=[
    ('Imputer', SimpleImputer(strategy='median')),
    ('Scaling', RobustScaler())
])
num_pipeline

In [ ]:
from sklearn.preprocessing import FunctionTransformer

log_pipeline = Pipeline(steps=[
    ('Log', FunctionTransformer(np.log1p, feature_names_out='one-to-one')),
    ('Scaling', RobustScaler())
])
log_pipeline

- Categorical Columns 

In [ ]:
cat_cols = x.select_dtypes(include= 'str').columns
cat_cols

In [ ]:
ohe_cols = ['CODE_GENDER', 'NAME_FAMILY_STATUS', 'NAME_INCOME_TYPE', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_HOUSING_TYPE',
            'NAME_CONTRACT_TYPE', 'NAME_CLIENT_TYPE']

ord_cols = ['NAME_EDUCATION_TYPE']

bin_cols = ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']

In [ ]:
from sklearn.preprocessing import OneHotEncoder

ohe_pipeline = Pipeline( [
    ('Imputer', SimpleImputer(strategy='most_frequent')),
    ('OHE', OneHotEncoder(sparse_output= False, drop= 'first', handle_unknown= 'ignore'))
])
ohe_pipeline

In [ ]:
x['NAME_EDUCATION_TYPE'].unique()

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

edu_order = ['Lower secondary', 'Secondary / secondary special', 'Incomplete higher',
             'Higher education', 'Academic degree']

ord_pipeline = Pipeline(steps=[
    ('Ord', OrdinalEncoder(categories=[edu_order]))
])
ord_pipeline

In [ ]:
from category_encoders import BinaryEncoder

bin_pipeline = Pipeline(steps=[
    ('Imputer', SimpleImputer(strategy='constant', fill_value='Other')),
    ('BE', BinaryEncoder())
])
bin_pipeline

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessing = ColumnTransformer(transformers=[
    ('Num Pipeline', num_pipeline, scal_cols),
    ('Log Pipeline', log_pipeline, log_cols),
    ('OHE Pipeline', ohe_pipeline, ohe_cols),
    ('Ordinal Pipeline', ord_pipeline, ord_cols),
    ('Binary Pipeline', bin_pipeline, bin_cols)
], remainder='passthrough') # passed through as-is (remainder)
preprocessing

### Select The Best Model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import  RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.model_selection import cross_validate

models = [
    ('Logistic Regression', LogisticRegression(random_state=42)),
    #('KNN', KNeighborsClassifier()),
    #('SVM', SVC(random_state= 42)),
    ('Naive Bayes', GaussianNB()),
    ('Decision Tree Classifier', DecisionTreeClassifier(random_state= 42)),
    ('Random Forest Classifier', RandomForestClassifier(random_state= 42)),
    ('XGB Classifier', XGBClassifier(random_state= 42))
]

for model in models:

    model_pipeline = Pipeline(steps= [('Preprocessing', preprocessing), ('Model', model[1])])

    cv_result = cross_validate(model_pipeline, x, y, scoring= ['accuracy', 'precision', 'recall', 'f1'], cv= 5, return_train_score= True, n_jobs= -2)

    print(model[0])

    # Accuracy Score
    print(f"Train Accuracy: {cv_result['train_accuracy'].mean().round(4) * 100:.2f}%")
    print(f"Test Accuracy: {cv_result['test_accuracy'].mean().round(4) * 100:.2f}%")

    # Precision Score
    print(f"Train Precision: {cv_result['train_precision'].mean().round(4) * 100:.2f}%")
    print(f"Test Precision: {cv_result['test_precision'].mean().round(4) * 100:.2f}%")

    # Recall Score
    print(f"Train Recall: {cv_result['train_recall'].mean().round(4) * 100:.2f}%")
    print(f"Test Recall: {cv_result['test_recall'].mean().round(4) * 100:.2f}%")

    # F1 Score
    print(f"Train F1 Score: {cv_result['train_f1'].mean().round(4) * 100:.2f}%")
    print(f"Test F1 Score: {cv_result['test_f1'].mean().round(4) * 100:.2f}%")

    print('-' * 50)

In [ ]:
ratio = (y == 0).sum() / (y == 1).sum()
ratio

In [ ]:
models = [
    ('Logistic Regression', LogisticRegression(random_state=42, class_weight='balanced')),
    #('KNN', KNeighborsClassifier()),
    #('SVM', SVC(random_state= 42)),
    ('Naive Bayes', GaussianNB()),
    ('Decision Tree Classifier', DecisionTreeClassifier(random_state= 42, class_weight='balanced')),
    ('Random Forest Classifier', RandomForestClassifier(random_state= 42, class_weight='balanced')),
    ('XGB Classifier', XGBClassifier(random_state= 42, scale_pos_weight=ratio))
]

for model in models:

    model_pipeline = Pipeline(steps= [('Preprocessing', preprocessing), ('Model', model[1])])

    cv_result = cross_validate(model_pipeline, x, y, scoring= ['accuracy', 'precision', 'recall', 'f1'], cv= 5, return_train_score= True, n_jobs= -1)

    print(model[0])

    # Accuracy Score
    print(f"Train Accuracy: {cv_result['train_accuracy'].mean().round(4) * 100:.2f}%")
    print(f"Test Accuracy: {cv_result['test_accuracy'].mean().round(4) * 100:.2f}%")

    # Precision Score
    print(f"Train Precision: {cv_result['train_precision'].mean().round(4) * 100:.2f}%")
    print(f"Test Precision: {cv_result['test_precision'].mean().round(4) * 100:.2f}%")

    # Recall Score
    print(f"Train Recall: {cv_result['train_recall'].mean().round(4) * 100:.2f}%")
    print(f"Test Recall: {cv_result['test_recall'].mean().round(4) * 100:.2f}%")

    # F1 Score
    print(f"Train F1 Score: {cv_result['train_f1'].mean().round(4) * 100:.2f}%")
    print(f"Test F1 Score: {cv_result['test_f1'].mean().round(4) * 100:.2f}%")

    print('-' * 50)

- **Without class weights**, most models predicted "Approved" for almost everyone (Recall 15–22%, below the 0.3 requirement), since only 15.4% of applications are refused.
- **With `class_weight='balanced'` / `scale_pos_weight`**, Logistic Regression and XGBoost passed (Precision ≥ 0.31, Recall ≥ 0.75). **XGBoost has the best F1 (46.7%)** with little overfitting.

### Hyperparameter Tuning

- Logistic Regression

In [ ]:
from sklearn.model_selection import GridSearchCV

lr_pipeline = Pipeline(steps=[('Preprocessing', preprocessing),
                              ('Model', LogisticRegression(random_state=42, class_weight='balanced', max_iter= 1000))])

lr_param_grid = {
    'Model__C': [0.001, 0.01, 0.1, 1, 10, 100],
    'Model__penalty': ['l2']
}

grid_search_lr = GridSearchCV(estimator=lr_pipeline, param_grid=lr_param_grid,
                              scoring=['f1', 'precision', 'recall'], refit='f1',
                              cv=5, n_jobs=-1, return_train_score=True)

grid_search_lr.fit(x, y)

In [ ]:
grid_search_lr.best_params_

In [ ]:
grid_search_lr.best_score_

- XGBoost

In [ ]:
xgb_pipeline = Pipeline(steps=[('Preprocessing', preprocessing),
                               ('Model', XGBClassifier(random_state=42, scale_pos_weight=ratio))])

xgb_param_grid = {
    'Model__n_estimators': [100, 200],
    'Model__max_depth': [3, 5, 7],
    'Model__learning_rate': [0.03, 0.1]
}

grid_search_xgb = GridSearchCV(estimator=xgb_pipeline, param_grid=xgb_param_grid,
                               scoring=['f1', 'precision', 'recall'], refit='f1',
                               cv=5, n_jobs=-2, return_train_score=True)

grid_search_xgb.fit(x, y)

In [ ]:
grid_search_xgb.best_params_

In [ ]:
grid_search_xgb.best_score_

In [ ]:
def show_best(grid):
    best = grid.best_index_
    res = grid.cv_results_
    print(grid.best_params_)
    for m in ['f1', 'precision', 'recall']:
        print(f"{m:10} Train: {res['mean_train_' + m][best]*100:.2f}%   Test: {res['mean_test_' + m][best]*100:.2f}%")
    print('-' * 50)

show_best(grid_search_lr)
show_best(grid_search_xgb)

- **Tuning (GridSearchCV, cv = 5, scoring = F1):** Logistic Regression F1 stayed ~44.7% for all `C` values. XGBoost best `learning_rate = 0.1`, `max_depth = 7`, `n_estimators = 200` → **F1 = 47.00%, Precision = 34.24%, Recall = 74.91%**, with small overfitting → **XGBoost selected**.

## Create Final Model

In [ ]:
xgboost_pipeline = Pipeline(steps=[('Preprocessing', preprocessing),
                                   ('Model', XGBClassifier(random_state=42, scale_pos_weight=ratio,
                                                           learning_rate= 0.1,
                                                           max_depth= 7,
                                                           n_estimators= 200))])
xgboost_pipeline

### Fit Final model to data

In [ ]:
xgboost_pipeline.fit(x, y)

### Save Model

In [ ]:
import joblib

# Save final model + input columns (for the Streamlit app)
joblib.dump(xgboost_pipeline, 'XGBoost_Approval_Pipeline.pkl')
joblib.dump(x.columns.tolist(), 'Inputs.pkl')

### Test Model

In [ ]:
model = joblib.load('XGBoost_Approval_Pipeline.pkl')

sample = x.sample(5, random_state=42)
print(model.predict(sample))
print(model.predict_proba(sample)[:, 1].round(3))
print(y.loc[sample.index].values)

## Feature Importance

In [ ]:
# Embedded feature selection: XGBoost importance of each feature
feature_names = xgboost_pipeline.named_steps['Preprocessing'].get_feature_names_out()
importances = xgboost_pipeline.named_steps['Model'].feature_importances_

fi = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
fi['Feature'] = fi['Feature'].str.split('__').str[-1]      # remove pipeline prefix
fi = fi.sort_values('Importance', ascending=False)

fi.head(20)

In [ ]:
px.bar(fi.head(20), x='Importance', y='Feature', orientation='h',
       title='Top 20 Features (XGBoost)').update_yaxes(autorange='reversed')

- **Embedded feature selection (XGBoost importance):** `NAME_CONTRACT_TYPE` is by far the most important feature: consumer loans are refused only 4.9% of the time (secured by the goods) vs ~27% for cash and revolving loans.
- `NAME_CLIENT_TYPE` comes second: new clients are refused 1.9% vs 20.9% for repeaters.
- Then `CNT_PAYMENT`, `EXT_SOURCE_3` and the documents. The new features `PAYMENT_TO_INCOME` and `DOWN_PAYMENT_RATIO` are in the top 20.